# TT-11 — Linear Regression: Định giá nhà 


In [ ]:
# 1. Nạp dữ liệu
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing(as_frame=True)
df = data.frame.rename(columns={"MedHouseVal": "target"})
df.describe().T


In [ ]:
# Phát hiện outlier cực đoan ở AveOccup, AveRooms, AveBedrms 
outlier_cols = ["AveRooms", "AveBedrms", "AveOccup"]
print(df[outlier_cols].describe(percentiles=[.95, .99, .999]))

# Clip theo phân vị 99 để loại lỗi dữ liệu vùng dân cư đặc biệt
for c in outlier_cols:
    cap = df[c].quantile(0.99)
    df[c] = df[c].clip(upper=cap)


In [ ]:
# 2. Phát hiện nhãn bị cắt ngọn ở 5.0 (bẫy dữ liệu #1)
n_capped = (df["target"] >= 5.0).sum()
print(f"Số căn nhà bị dồn vào đúng giá 5.0: {n_capped} ({n_capped/len(df):.2%} tổng số)")
# => phải nêu trong phần hạn chế: model không dự đoán được nhà đắt hơn 500k USD


## 3. EDA

In [ ]:
# Scatter MedInc vs giá
plt.figure(figsize=(6, 4))
plt.scatter(df["MedInc"], df["target"], alpha=0.15, s=8)
plt.xlabel("MedInc (thu nhập trung vị)"); plt.ylabel("Giá (100k USD)")
plt.title("MedInc vs Giá nhà")
plt.show()


In [ ]:
# Heatmap tương quan giữa các đặc trưng và target
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Ma trận tương quan")
plt.show()


In [ ]:
# Bản đồ giá theo toạ độ (Latitude/Longitude) — trực quan hoá quan hệ phi tuyến
plt.figure(figsize=(7, 6))
sc = plt.scatter(df["Longitude"], df["Latitude"], c=df["target"],
                  cmap="viridis", s=6, alpha=0.5)
plt.colorbar(sc, label="Giá (100k USD)")
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.title("Giá nhà theo toạ độ địa lý")
plt.show()


## 4. Baseline & Linear Regression cơ bản

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

X = df.drop(columns="target")
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def eval_model(y_true, y_pred, name):
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    print(f"{name:20s} RMSE={rmse:.3f}  MAE={mae:.3f}  R2={r2:.3f}  MAPE={mape:.2%}")
    return rmse, mae, r2, mape


In [ ]:
# Baseline: dự đoán bằng trung bình
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
_ = eval_model(y_test, baseline.predict(X_test), "Baseline (mean)")


In [ ]:
# Linear Regression (đã chuẩn hoá đặc trưng)
pipe = Pipeline([("scale", StandardScaler()), ("lr", LinearRegression())])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
_ = eval_model(y_test, y_pred, "Linear Regression")


## 5. Kiểm tra 4 giả định của hồi quy tuyến tính

In [ ]:
# Residual plot: phần dư vs giá trị dự đoán -> kiểm tra giả định (3) phương sai đều
residuals = y_test - y_pred
plt.figure(figsize=(6, 4))
plt.scatter(y_pred, residuals, alpha=0.2, s=8)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Giá dự đoán"); plt.ylabel("Phần dư (residual)")
plt.title("Residual plot — kiểm tra hiện tượng hình phễu (heteroscedasticity)")
plt.show()


In [ ]:
# Q-Q plot: kiểm tra giả định (4) phần dư phân phối chuẩn
import scipy.stats as stats
plt.figure(figsize=(5, 5))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q-Q plot phần dư")
plt.show()


**Nhận xét giả định:**
- (1) Tuyến tính: scatter MedInc–giá có xu hướng thẳng nhưng nhiễu lớn ở vùng giá cao (do nhãn bị cắt ngọn).
- (2) Độc lập: các quan sát theo hộ gia đình, giả định hợp lý ở mức chấp nhận được.
- (3) Phương sai đều: residual plot thường có dạng loe ra (hình phễu) khi giá tăng → **vi phạm**.
- (4) Sai số chuẩn: Q-Q plot lệch đuôi phải do cụm nhà bị cắt ngọn ở 5.0 → **vi phạm nhẹ**.

## 6. Thử log-transform target để giảm phương sai không đều

In [ ]:
y_train_log, y_test_log = np.log1p(y_train), np.log1p(y_test)
pipe_log = Pipeline([("scale", StandardScaler()), ("lr", LinearRegression())])
pipe_log.fit(X_train, y_train_log)

y_pred_log = np.expm1(pipe_log.predict(X_test))  # quy đổi ngược để so RMSE cùng đơn vị
_ = eval_model(y_test, y_pred_log, "Linear Regression (log-target)")

residuals_log = y_test - y_pred_log
plt.figure(figsize=(6, 4))
plt.scatter(y_pred_log, residuals_log, alpha=0.2, s=8)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Giá dự đoán (quy đổi từ log)"); plt.ylabel("Phần dư")
plt.title("Residual plot sau log-transform")
plt.show()
# Kết luận: log-transform thường giúp residual đều hơn ở vùng giá thấp/trung bình,
# nhưng không giải quyết được gốc rễ vấn đề cắt ngọn ở giá cao.


## 7. Kiểm tra đa cộng tuyến (VIF)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = X_train.copy()
vif = pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).sort_values("VIF", ascending=False)
print(vif)
# Kỳ vọng: AveRooms và AveBedrms có VIF cao do tương quan chặt với nhau


## 8. Feature engineering

In [ ]:
def add_features(X):
    X = X.copy()
    X["rooms_per_household"] = X["AveRooms"] / X["AveOccup"].replace(0, np.nan)
    # khoảng cách Euclid gần đúng tới San Francisco và Los Angeles
    sf, la = (37.77, -122.42), (34.05, -118.24)
    X["dist_to_SF"] = np.sqrt((X["Latitude"] - sf[0])**2 + (X["Longitude"] - sf[1])**2)
    X["dist_to_LA"] = np.sqrt((X["Latitude"] - la[0])**2 + (X["Longitude"] - la[1])**2)
    return X.fillna(X.median(numeric_only=True))

X_train_fe, X_test_fe = add_features(X_train), add_features(X_test)

pipe_fe = Pipeline([("scale", StandardScaler()), ("lr", LinearRegression())])
pipe_fe.fit(X_train_fe, y_train)
_ = eval_model(y_test, pipe_fe.predict(X_test_fe), "Linear Regression + FE")


## 9. Bảng hệ số đã chuẩn hoá & diễn giải

In [ ]:
he_so = pd.Series(pipe_fe["lr"].coef_, index=X_train_fe.columns).sort_values(key=abs, ascending=False)
print(he_so)

top3 = he_so.head(3)
print("\n3 yếu tố ảnh hưởng mạnh nhất đến giá nhà (theo độ lớn hệ số chuẩn hoá):")
for feat, val in top3.items():
    huong = "tăng" if val > 0 else "giảm"
    print(f" - {feat}: hệ số {val:.3f} -> đặc trưng này {huong} giá dự đoán khi nó tăng")
# Lưu ý: đây là mối liên hệ thống kê, KHÔNG phải quan hệ nhân quả


## 10. So sánh với Ridge (TT-12) và Random Forest (TT-17)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

ridge = Pipeline([("scale", StandardScaler()), ("ridge", Ridge(alpha=1.0))])
ridge.fit(X_train_fe, y_train)
_ = eval_model(y_test, ridge.predict(X_test_fe), "Ridge")

rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train_fe, y_train)
_ = eval_model(y_test, rf.predict(X_test_fe), "Random Forest")
# Random Forest thường vượt trội hơn về RMSE/R2 nhưng đánh đổi tính giải thích được


## 11. Hạn chế của model

- **Nhãn bị cắt ngọn ở 5.0**: model không bao giờ dự đoán được nhà giá trên 500k USD một cách chính xác.
- **Quan hệ toạ độ–giá là phi tuyến**: Linear Regression bắt kém quan hệ này dù đã thêm đặc trưng khoảng cách; các model cây (Random Forest) xử lý tốt hơn nhiều.
- **Đánh đổi**: chênh lệch R² giữa Linear Regression (~0.58–0.61) và Random Forest (~0.80) chính là **cái giá của tính giải thích được** — phù hợp khi nghiệp vụ bắt buộc giải thích cho khách hàng.